# EDA and cleaning data for silver layer

In [0]:
df = spark.sql("FROM supply_chain_live.bronze.raw_supply_chain")
df.display()

In [0]:
metadata = spark.sql("FROM supply_chain_live.bronze.metadata")
metadata.display()

In [0]:
print(df.columns)

In [0]:
df.select(
    "Customer Email",
    "Customer Country",
    "Benefit per order",
    "shipping date (DateOrders)",
).limit(10).display()

In [0]:
df.select("Product Description").distinct().display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count the number of null values in each column
null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

# Convert the result to a dictionary so that we can loop through it
null_counts = null_counts.collect()[0].asDict()

# Only keep columns with null values
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

## Summary of cleaning steps

We note down what to be cleaned based on looking at the data, reading metadata and doing some EDA. This list is not comprehensive, I don't want to take away all the fun for you :)

I will leave it to you to continue clean this data

**Drop**
- Customer Email (data is masked)
- Customer Password
- Product Description (all are nulls) 

**Rounding errors**
- Order Item Product Price -> 2 decimals
- Order Item Product Price -> 2 decimals
- Benefit per order -> 2 decimals
- Sales per customer -> 2 decimals 
...

**Customer country** 
- EE. UU. -> United States as it is the spanish abbreviation

**Nulls**
- fill Customer Lname with missing
- fill Customer Zipcode with missing
- fill Order Zipcode with missing

**Data type**
- shipping date - string -> Timestamp

**Derived columns** - will not remove them in silver layer, but will compute them in Gold layer instead
- Order Item Total - derived from (Order Item Quantity)*(Order Item Product Price)*(1-Order Item Discount Rate), calculate and test this 
...

**Rename columns**
- use snake_case convention instead (this is a design choice)


In [0]:
import re 

#     Customer   Email    -> customer_email
def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

to_snake_case(" Customer    Email  AcCount ")

In [0]:
df_cleaned_columns = rename_columns_to_snake_case(df)
df_cleaned_columns.display()


more cleaning

In [0]:
df_cleaned_columns.select("shipping_date_(dateorders)").limit(1).display()

In [0]:
from pyspark.sql.functions import to_timestamp, col

# 6/19/2017 4:41
df_cleaned_columns.withColumn(
    "shipping_date", to_timestamp("shipping_date_(dateorders)", "M/d/yyyy H:mm")
)

.select("shipping_date", "shipping_date_(dateorders)").display()